In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re

# Display settings
pd.set_option('display.max_colwidth', 200)

# Load the dataset
df = pd.read_csv("../data/Resume.csv")

# Show basic info
print("Shape of dataset:", df.shape)
df.head()


Shape of dataset: (2484, 4)


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management. ...,"<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME500375979"" style=""\n padding-top:0px;\n ""> <div class...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS Summary Versatile media professional with background in Communications, Marketing, Human Resources and Technology. Experience 09/201...","<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME911808366"" style=""padding-top:0px;""> <div class=""paragraph PA...",HR
2,33176873,"HR DIRECTOR Summary Over 20 years experience in recruiting, 15 plus years in Human Resources Executive Management, 5 years of HRIS development and maintenance 4 years work...","<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME1008511259"" style=""padding-top:0px;""> <div class=""paragraph P...",HR
3,27018550,"HR SPECIALIST Summary Dedicated, Driven, and Dynamic with over 20 years of customer service expertise. Motivated to maintain customer satisfaction and contribute to company succe...","<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME992636658"" style=""padding-top:0px;""> <div class=""paragraph PA...",HR
4,17812897,HR MANAGER Skill Highlights HR SKILLS HR Department Startup Three New Organization Startups Employment Law FMLA/ADA/EEO/WC Mediation & Advocacy HR Policies & Proce...,"<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME666809417"" style=""padding-top:0px;""> <div class=""paragraph PA...",HR


In [2]:
# Check column names
df.columns


Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')

In [3]:
# View one sample resume text
print(df.iloc[0])


ID                                                                                                                                                                                                            16852973
Resume_str              HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management. ...
Resume_html    <div class="fontsize fontface vmargins hmargins linespacing pagesize" id="document"> <div class="section firstsection" id="SECTION_NAME500375979" style="\n      padding-top:0px;\n    "> <div class...
Category                                                                                                                                                                                                            HR
Name: 0, dtype: object


In [4]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_resume(text):
    text = text.lower()                          # lowercase
    text = re.sub(r'<.*?>', ' ', text)           # remove HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)        # remove numbers & symbols
    text = re.sub(r'\s+', ' ', text)             # remove extra spaces
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\muluk\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [5]:
df['clean_resume'] = df['Resume_str'].apply(clean_resume)
df[['Resume_str', 'clean_resume']].head(2)


,Resume_str,clean_resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management. ...,hr administrator marketing associate hr administrator summary dedicated customer service manager years experience hospitality customer service management respected builder leader customer focused ...
1,"HR SPECIALIST, US HR OPERATIONS Summary Versatile media professional with background in Communications, Marketing, Human Resources and Technology. Experience 09/201...",hr specialist us hr operations summary versatile media professional background communications marketing human resources technology experience current hr specialist us hr operations company name ci...


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['clean_resume'])

print(X.shape)


(2484, 3000)


In [7]:
# Example Job Description
job_description = """
Looking for a Data Scientist with experience in Python, Machine Learning,
Data Analysis, SQL, and statistics. Experience with NLP and data visualization
is a plus.
"""

# Clean the job description
job_clean = clean_resume(job_description)

# Convert job description to TF-IDF
job_tfidf = tfidf.transform([job_clean])


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between job description and all resumes
similarity_scores = cosine_similarity(job_tfidf, X).flatten()

# Add scores to dataframe
df['match_score'] = similarity_scores

# Sort resumes by match score (highest first)
ranked_df = df.sort_values(by='match_score', ascending=False)

# Show top 5 ranked resumes
ranked_df[['ID', 'Category', 'match_score']].head()


,ID,Category,match_score
1762,12011623,ENGINEERING,0.368781
1218,21156767,CONSULTANT,0.361700
1339,18448085,AUTOMOBILE,0.288123
331,18067556,INFORMATION-TECHNOLOGY,0.269557
1142,30863060,CONSULTANT,0.257861


In [9]:
# Save ranked resumes
ranked_df.to_csv("outputs/ranked_resumes.csv", index=False)

print("Ranked resumes saved successfully!")


OSError: Cannot save file into a non-existent directory: 'outputs'

In [10]:
import os

# Create outputs folder if it doesn't exist
os.makedirs("outputs", exist_ok=True)


In [11]:
ranked_df.to_csv("outputs/ranked_resumes.csv", index=False)
print("Ranked resumes saved successfully!")


Ranked resumes saved successfully!


In [12]:
job_skills = [
    "python", "machine learning", "data analysis",
    "sql", "excel", "statistics", "nlp"
]



In [13]:
def extract_skills(resume_text, skills_list):
    found_skills = []
    for skill in skills_list:
        if skill in resume_text:
            found_skills.append(skill)
    return found_skills


In [14]:
def find_skill_gap(resume_text, skills_list):
    present_skills = extract_skills(resume_text, skills_list)
    missing_skills = list(set(skills_list) - set(present_skills))
    return present_skills, missing_skills


In [15]:
df["present_skills"] = df["clean_resume"].apply(
    lambda x: extract_skills(x, job_skills)
)

df["missing_skills"] = df["clean_resume"].apply(
    lambda x: list(set(job_skills) - set(extract_skills(x, job_skills)))
)


In [16]:
skill_gap_df = df[["ID", "present_skills", "missing_skills"]]
skill_gap_df.to_csv("outputs/skill_gap_analysis.csv", index=False)

print("Skill gap analysis saved successfully!")


Skill gap analysis saved successfully!
